# CymbalGoal — Provisioning Prototype (v2)

**Measurement prototype, not lab content.** Answers three questions:

1. Can a notebook reach the cluster, and how?
2. How long does the full load take versus the **141 s** measured from a VM?
3. Does this surface show `NOTICE` output? (Task 3's BM25 instrumentation depends on it.)

**v2 changes, all learned from `cymbalflix_database_setup.ipynb`:**

- **AlloyDB Python Connector with `enable_iam_auth=True`** instead of raw `psycopg`. Your Google
  identity *is* your database identity — no password anywhere. v1 used `psycopg` straight at the
  private IP and timed out, because a non-VPC-attached runtime has no route to `10.188.244.2`.
- **Nothing hardcoded.** Project, region, cluster, instance and user are all discovered.
- **Server-side `gcloud alloydb clusters import`** for the relational load. Data flows GCS → AlloyDB
  directly instead of through this notebook, which matters when 300 runtimes do it at once.

In [ ]:
!pip install -q "google-cloud-alloydb-connector[pg8000]" pandas sqlalchemy 2>&1 | tail -1
print("deps ready")

## Discover everything

Nothing hardcoded. This has to survive being handed to any lab project without edits — a
hardcoded project ID is a lab that breaks for the next student.

In [ ]:
import subprocess, json, time, re

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip()

PROJECT   = sh("gcloud config get-value project")
USER      = sh("gcloud config get-value account")
PROJ_NUM  = sh(f"gcloud projects describe {PROJECT} --format='value(projectNumber)'")

# Find the cluster rather than assuming its name, so this works against any lab project.
clusters = json.loads(sh("gcloud alloydb clusters list --format=json") or "[]")
assert clusters, "no AlloyDB cluster found in this project"
c0        = clusters[0]
CLUSTER   = c0["name"].split("/")[-1]
REGION    = c0["name"].split("/locations/")[1].split("/")[0]

instances = json.loads(sh(f"gcloud alloydb instances list --cluster={CLUSTER} "
                          f"--region={REGION} --format=json") or "[]")
primary   = [i for i in instances if i.get("instanceType") == "PRIMARY"][0]
INSTANCE  = primary["name"].split("/")[-1]

INSTANCE_URI = f"projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}/instances/{INSTANCE}"
ALLOYDB_SA   = f"service-{PROJ_NUM}@gcp-sa-alloydb.iam.gserviceaccount.com"
DB_NAME      = "cymbalgoal"
GCS          = "gs://class-demo/alloydb-labs/cymbalgoal"

HAS_PUBLIC  = bool(primary.get("publicIpAddress"))
PRIVATE_IP  = primary.get("ipAddress")

for k, v in [("project", PROJECT), ("region", REGION), ("cluster", CLUSTER),
             ("instance", INSTANCE), ("user", USER), ("alloydb SA", ALLOYDB_SA),
             ("private ip", PRIVATE_IP), ("public ip", primary.get("publicIpAddress") or "none")]:
    print(f"  {k:12s} {v}")

## Connect

The connector needs a network path to whichever IP type you ask for:

- `IPTypes.PRIVATE` — requires this runtime to be **VPC-attached** to `cymbalgoal-network`.
- `IPTypes.PUBLIC` — requires the instance to have a public IP. What CymbalFlix uses.

Either way `enable_iam_auth=True` means no password is involved. We try private first and fall
back, so the output tells us which paths actually exist.

In [ ]:
from google.cloud.alloydb.connector import Connector, IPTypes
import pg8000, pandas as pd

connector = Connector()
IP_TYPE = None
TIMINGS = {}

def _try(ip_type):
    c = connector.connect(INSTANCE_URI, "pg8000", user=USER, db="postgres",
                          enable_iam_auth=True, ip_type=ip_type)
    cur = c.cursor(); cur.execute("SELECT version()"); v = cur.fetchone()[0]
    cur.close(); c.close(); return v

for name, t in [("PRIVATE", IPTypes.PRIVATE), ("PUBLIC", IPTypes.PUBLIC)]:
    try:
        t0 = time.time()
        v = _try(t)
        IP_TYPE = t
        print(f"  {name}: CONNECTED in {time.time()-t0:.1f}s")
        print("   ", v)
        break
    except Exception as e:
        print(f"  {name}: failed — {str(e)[:110]}")

assert IP_TYPE is not None, (
    "No path to the cluster.\n"
    "  PRIVATE failed -> runtime is not VPC-attached to cymbalgoal-network\n"
    "  PUBLIC  failed -> instance has no public IP\n"
    "Fix one of those before continuing."
)

In [ ]:
def conn(db="postgres", autocommit=True):
    c = connector.connect(INSTANCE_URI, "pg8000", user=USER, db=db,
                          enable_iam_auth=True, ip_type=IP_TYPE)
    c.autocommit = autocommit
    return c

def run(sql, db=DB_NAME, fetch=False, show_notices=False):
    c = conn(db); cur = c.cursor()
    cur.execute(sql)
    out = cur.fetchall() if fetch else None
    if show_notices:
        # Does THIS surface expose NOTICE? Task 3's BM25 instrumentation
        # (k1, b, document count, avg_length) arrives this way and nowhere else.
        for n in getattr(c, "notices", []) or []:
            print("    NOTICE:", str(n).strip())
    cur.close(); c.close()
    return out

from contextlib import contextmanager
@contextmanager
def phase(name):
    t0 = time.time(); print(f"--- {name} ---")
    yield
    TIMINGS[name] = round(time.time()-t0, 1); print(f"    {TIMINGS[name]} s\n")

print("identity check:", run("SELECT current_user, "
      "pg_has_role(current_user,'alloydbsuperuser','member')",
      db="postgres", fetch=True))

## Database, extensions, schema

Everything the startup VM did in steps 1–4.

`schema.sql` also creates the two ScaNN indexes on empty tables. We drop them by name and rebuild
after the load — measured slower by 45 s, taken deliberately so centroids train on the real
distribution rather than on zero rows. **No regex splitting of DDL.**

In [ ]:
with phase("01_database"):
    c = conn("postgres"); cur = c.cursor()
    cur.execute(f"SELECT 1 FROM pg_database WHERE datname='{DB_NAME}'")
    if cur.fetchone():
        cur.execute(f"DROP DATABASE {DB_NAME}")
    cur.execute(f"CREATE DATABASE {DB_NAME}")
    cur.close(); c.close()
    print(f"    {DB_NAME} created")

with phase("02_extensions"):
    for ext in ["vector","alloydb_scann","google_ml_integration","pg_textsearch"]:
        run(f"CREATE EXTENSION IF NOT EXISTS {ext}")
    for r in run("SELECT extname, extversion FROM pg_extension ORDER BY 1", fetch=True):
        print("   ", r[0], r[1])

with phase("03_schema"):
    schema = sh(f"gcloud storage cat {GCS}/schema.sql")
    run(schema)
    run("DROP INDEX IF EXISTS players_profile_embedding_scann_idx")
    run("DROP INDEX IF EXISTS clubs_profile_embedding_scann_idx")
    print("    schema applied, ScaNN indexes dropped")

## Pass 1 — server-side import

`gcloud alloydb clusters import` streams GCS → AlloyDB directly. The data never passes through this
notebook, which is what makes 300 concurrent students viable.

Two guards, both earned the hard way:

- The manifest's `column_order` is **DDL-derived** — for `players` and `clubs` it lists
  `profile_text` / `profile_embedding`, which the pass-1 CSVs don't carry. Subtract them.
- **Preflight the field count.** A list *longer* than the file errors loudly. A list of the same
  length in a different order loads *silently* into the wrong columns.

P-18: the import API is one table per call and permits one operation at a time, so these are serial.

In [ ]:
import gzip, csv, io
PASS2 = {"profile_text","profile_embedding"}
ORDER = ["competitions","clubs","players","games",
         "appearances","game_events","player_valuations","transfers"]

man = json.loads(sh(f"gcloud storage cat {GCS}/manifest.json"))
sf  = man.get("staged_files")
items = sf.items() if isinstance(sf, dict) else [(f.get("name"), f) for f in sf]
COLS = {}
for k, v in items:
    if isinstance(v, dict) and v.get("column_order"):
        t = str(k).split("/")[-1].replace(".csv.gz","").replace(".csv","")
        COLS[t] = [c for c in v["column_order"] if c not in PASS2]

with phase("04_preflight"):
    for t in ORDER:
        assert t in COLS, f"{t}: no column_order in manifest — never guess"
        raw  = subprocess.run(["gcloud","storage","cat",f"{GCS}/{t}.csv.gz"],
                              capture_output=True).stdout
        head = gzip.decompress(raw)[:200_000].decode("utf-8","replace")
        n_file = len(next(csv.reader(io.StringIO(head))))
        assert len(COLS[t]) == n_file, f"{t}: list={len(COLS[t])} file={n_file} — REFUSING"
        print(f"    {t:20s} {n_file} columns OK")

In [ ]:
with phase("05_pass1_import"):
    for t in ORDER:
        t0 = time.time()
        cmd = (f"gcloud alloydb clusters import {CLUSTER} --region={REGION} "
               f"--database={DB_NAME} --gcs-uri={GCS}/{t}.csv.gz "
               f"--csv --table={t} --columns={','.join(COLS[t])} --quiet")
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        n = run(f"SELECT count(*) FROM {t}", fetch=True)[0][0]
        status = "ok" if r.returncode == 0 else f"FAILED rc={r.returncode}"
        print(f"    {t:20s} {n:>9,} rows  {time.time()-t0:6.1f}s  {status}")
        if r.returncode != 0:
            print("      ", (r.stderr or "")[:400])

## Pass 2 — profiles

`load_profiles.sql` uses `\copy ... FROM PROGRAM`, a psql meta-command that does not exist here, so
we run the same two `UPDATE`s ourselves.

⚠️ Its assertion covers **players only** — `n_clubs` is computed, printed, never checked. A zero-row
clubs load passes silently. We add the missing check.

In [ ]:
with phase("06_pass2"):
    for t in ["players","clubs"]:
        t0 = time.time()
        r = subprocess.run(
            f"gcloud alloydb clusters import {CLUSTER} --region={REGION} "
            f"--database={DB_NAME} --gcs-uri={GCS}/{t}_profiles.csv.gz "
            f"--csv --table=_{t}_p --columns={'player_id' if t=='players' else 'club_id'},"
            f"profile_text,profile_embedding --quiet",
            shell=True, capture_output=True, text=True)
        print(f"    {t} import rc={r.returncode} {time.time()-t0:.1f}s")
        if r.returncode != 0:
            print("      ", (r.stderr or "")[:300])

⚠️ The cell above will likely fail: the import API needs the **target table to exist**, and
`_players_p` does not. If so, create the staging tables first and re-run — that requirement is
itself a finding worth recording, because it means pass 2 cannot be a pure import.

In [ ]:
# Fallback: create staging tables, then import into them.
with phase("06b_pass2_staged"):
    run("CREATE TABLE IF NOT EXISTS _players_p (player_id INTEGER, profile_text TEXT, "
        "profile_embedding VECTOR(3072))")
    run("CREATE TABLE IF NOT EXISTS _clubs_p (club_id INTEGER, profile_text TEXT, "
        "profile_embedding VECTOR(3072))")
    for t, key in [("players","player_id"), ("clubs","club_id")]:
        t0 = time.time()
        r = subprocess.run(
            f"gcloud alloydb clusters import {CLUSTER} --region={REGION} "
            f"--database={DB_NAME} --gcs-uri={GCS}/{t}_profiles.csv.gz "
            f"--csv --table=_{t}_p --columns={key},profile_text,profile_embedding --quiet",
            shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"    {t}: import failed —", (r.stderr or "")[:300]); continue
        run(f'''UPDATE {t} x SET profile_text = s.profile_text,
                profile_embedding = s.profile_embedding
                FROM _{t}_p s WHERE x.{key} = s.{key}''')
        n = run(f"SELECT count(*) FROM {t} WHERE profile_embedding IS NOT NULL", fetch=True)[0][0]
        print(f"    {t:10s} {n:>8,} profiles  {time.time()-t0:.1f}s")
    run("DROP TABLE IF EXISTS _players_p"); run("DROP TABLE IF EXISTS _clubs_p")

p = run("SELECT count(*) FROM players WHERE profile_embedding IS NOT NULL", fetch=True)[0][0]
k = run("SELECT count(*) FROM clubs   WHERE profile_embedding IS NOT NULL", fetch=True)[0][0]
assert p >= 13304, f"player profile load short: {p}"
assert k >= 790,  f"club profile load short: {k}"      # the check load_profiles.sql lacks
print(f"    asserted: {p:,} players / {k:,} clubs")

## ScaNN, after the load

`num_leaves` ≈ √rows. **This is the phase most worth moving into student view** — ScaNN is a
featured product currently buried in a shell script nobody reads.

In [ ]:
with phase("07_scann"):
    for t, leaves in [("players",116), ("clubs",28)]:
        t0 = time.time()
        run(f"SET maintenance_work_mem='4GB'")
        run(f'''CREATE INDEX {t}_profile_embedding_scann_idx ON {t}
                USING scann (profile_embedding cosine)
                WITH (num_leaves={leaves}, quantizer='sq8')''')
        print(f"    {t:10s} {time.time()-t0:.1f}s")
    run("ANALYZE players"); run("ANALYZE clubs")

## Does this surface show NOTICE output?

The open question. If yes, Task 3 gets `k1`, `b`, document count and average length for free.

In [ ]:
with phase("08_notice_check"):
    run("DROP INDEX IF EXISTS players_profile_text_bm25_idx")
    run('''CREATE INDEX players_profile_text_bm25_idx ON players
           USING bm25 (profile_text) WITH (text_config = 'english')''', show_notices=True)
    print("    ^ expect: 13439 documents, avg_length=161.68")
    run("DROP INDEX players_profile_text_bm25_idx")   # Task 3 builds it, not us

In [ ]:
run('''CREATE TABLE IF NOT EXISTS provisioning_status (
         finished_at timestamptz PRIMARY KEY DEFAULT now(),
         players bigint, clubs bigint, appearances bigint)''')
run('''INSERT INTO provisioning_status (players, clubs, appearances)
       SELECT (SELECT count(*) FROM players WHERE profile_embedding IS NOT NULL),
              (SELECT count(*) FROM clubs   WHERE profile_embedding IS NOT NULL),
              (SELECT count(*) FROM appearances)''')
print(run("SELECT * FROM provisioning_status", fetch=True))

print("\n=========== PHASE TIMINGS ===========")
for k_, v_ in TIMINGS.items():
    print(f"  {k_:24s} {v_:8.1f} s")
print(f"  {'TOTAL':24s} {sum(TIMINGS.values()):8.1f} s")
print("=====================================")
print("VM baseline: pass1 82s + pass2 14s + scann 45s = 141 s")
print(f"IP path used: {IP_TYPE}")